# Anime Data Analysis

This notebook explores an anime dataset, performing feature engineering to extract episode counts and airing durations, followed by an analysis of scores and rankings.

### 1. Imports and Data Loading

We start by importing the necessary libraries and loading the `anime.csv` dataset.

In [8]:
import pandas as pd
from dateutil.relativedelta import relativedelta
from datetime import datetime

# Load the dataset
anime_df = pd.read_csv('anime.csv')
anime_df.head()

,Rank,Title,Score
0,1,Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...,9.10
1,2,"Steins;GateTV (24 eps)Apr 2011 - Sep 20112,473...",9.07
2,3,Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 -...,9.06
3,4,"Gintama°TV (51 eps)Apr 2015 - Mar 2016605,113 ...",9.06
4,5,Shingeki no Kyojin Season 3 Part 2TV (10 eps)A...,9.05


### 2. Feature Engineering: Extracting Episode Count

The episode count is currently embedded within the `Title` string in parentheses. We define a function to extract this information and convert it to an integer type.

In [9]:
def eps_extract(title):
    """Extracts the number of episodes from the title string."""
    check = False
    data = ''
    for i in title:
        if i == ')':
            check = False
            return data
        if check:
            data = data + i
        if i == '(':
            check = True
    return data

# Apply extraction and cleaning
anime_df['Episodes'] = anime_df["Title"].apply(eps_extract)
anime_df["Episodes" ] = anime_df["Episodes"].str.replace(' eps', '')

# Convert to integer
anime_df['Episodes'] = anime_df["Episodes"].astype(int)

print(f"Datatype of Episodes: {anime_df['Episodes'].dtype}")
anime_df.head()

Datatype of Episodes: int64


,Rank,Title,Score,Episodes
0,1,Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...,9.10,64
1,2,"Steins;GateTV (24 eps)Apr 2011 - Sep 20112,473...",9.07,24
2,3,Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 -...,9.06,13
3,4,"Gintama°TV (51 eps)Apr 2015 - Mar 2016605,113 ...",9.06,51
4,5,Shingeki no Kyojin Season 3 Part 2TV (10 eps)A...,9.05,10


### 3. Feature Engineering: Extracting Airing Time Span

We also extract the string representing the airing period of each anime.

In [10]:
def extract_time(title):
    """Extracts the time span suffix from the title."""
    data = ''
    for i in range(len(title)):
        if title[i] == ')':
            # Extract characters after the first closing parenthesis
            data = title[i+1: i+20].strip()
            break
    return data

anime_df["Time"] = anime_df["Title"].apply(extract_time)
anime_df.head()

,Rank,Title,Score,Episodes,Time
0,1,Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...,9.10,64,Apr 2009 - Jul 2010
1,2,"Steins;GateTV (24 eps)Apr 2011 - Sep 20112,473...",9.07,24,Apr 2011 - Sep 2011
2,3,Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 -...,9.06,13,Oct 2022 - Dec 2022
3,4,"Gintama°TV (51 eps)Apr 2015 - Mar 2016605,113 ...",9.06,51,Apr 2015 - Mar 2016
4,5,Shingeki no Kyojin Season 3 Part 2TV (10 eps)A...,9.05,10,Apr 2019 - Jul 2019


### 4. Data Analysis: Scoring and Popularity

In this section, we identify the top-performing anime based on their scores and find the ones with the most episodes.

In [11]:
# Finding the highest score
max_score = anime_df["Score"].max()
highest_score_anime = anime_df[anime_df["Score"] == max_score]["Title"].iloc[0]
print(f"The anime with the highest score ({max_score}) is: {highest_score_anime}")

print("\nTop 5 Highest Scoring Anime:")
display(anime_df.nlargest(5, "Score"))

The anime with the highest score (9.1) is: Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr 2009 - Jul 20103,218,472 membersManga StoreVolume 1€4.58Preview

Top 5 Highest Scoring Anime:


,Rank,Title,Score,Episodes,Time
0,1,Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...,9.10,64,Apr 2009 - Jul 2010
1,2,"Steins;GateTV (24 eps)Apr 2011 - Sep 20112,473...",9.07,24,Apr 2011 - Sep 2011
2,3,Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 -...,9.06,13,Oct 2022 - Dec 2022
3,4,"Gintama°TV (51 eps)Apr 2015 - Mar 2016605,113 ...",9.06,51,Apr 2015 - Mar 2016
4,5,Shingeki no Kyojin Season 3 Part 2TV (10 eps)A...,9.05,10,Apr 2019 - Jul 2019


In [12]:
# Highest episode count
highest_eps = anime_df["Episodes"].max()
highest_eps_anime = anime_df[anime_df["Episodes"] == highest_eps]
print(f"Anime with the highest episode count ({highest_eps}):")
display(highest_eps_anime)

print("\nTop 5 Anime by Episode Count:")
display(anime_df.nlargest(5, "Episodes"))

Anime with the highest episode count (201):


,Rank,Title,Score,Episodes,Time
15,16,"GintamaTV (201 eps)Apr 2006 - Mar 20101,034,41...",8.94,201,Apr 2006 - Mar 2010



Top 5 Anime by Episode Count:


,Rank,Title,Score,Episodes,Time
15,16,"GintamaTV (201 eps)Apr 2006 - Mar 20101,034,41...",8.94,201,Apr 2006 - Mar 2010
7,8,Hunter x Hunter TV (148 eps)Oct 2011 - Sep 201...,9.04,148,Oct 2011 - Sep 2014
11,12,Ginga Eiyuu DensetsuOVA (110 eps)Jan 1988 - Ma...,9.02,110,Jan 1988 - Mar 1997
42,43,Hajime no IppoTV (75 eps)Oct 2000 - Mar 200255...,8.76,75,Oct 2000 - Mar 2002
24,25,"MonsterTV (74 eps)Apr 2004 - Sep 20051,041,081...",8.87,74,Apr 2004 - Sep 2005


### 5. Advanced Analysis: Longest Running Anime

We calculate the total duration in months for each anime based on the extracted time span.

In [13]:
def calculate_duration_months(time_str):
    """Calculates total months between two dates in 'Jan 2000' format."""
    try:
        start, end = time_str.split(' - ')
        start_date = datetime.strptime(start.strip(), '%b %Y')
        end_date = datetime.strptime(end.strip(), '%b %Y')
        r = relativedelta(end_date, start_date)
        # Including the start month in total count
        return r.years * 12 + r.months + 1
    except:
        return None

anime_df["Months"] = anime_df["Time"].apply(calculate_duration_months)
anime_df.sort_values("Months", ascending=False).head()

,Rank,Title,Score,Episodes,Time,Months
11,12,Ginga Eiyuu DensetsuOVA (110 eps)Jan 1988 - Ma...,9.02,110,Jan 1988 - Mar 1997,111
15,16,"GintamaTV (201 eps)Apr 2006 - Mar 20101,034,41...",8.94,201,Apr 2006 - Mar 2010,48
7,8,Hunter x Hunter TV (148 eps)Oct 2011 - Sep 201...,9.04,148,Oct 2011 - Sep 2014,36
32,33,Kingdom 3rd SeasonTV (26 eps)Apr 2020 - Oct 20...,8.81,26,Apr 2020 - Oct 2021,19
24,25,"MonsterTV (74 eps)Apr 2004 - Sep 20051,041,081...",8.87,74,Apr 2004 - Sep 2005,18
